## Library yang boleh

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Baca Data

In [5]:
df = pd.read_excel('Auto_MPG.xlsx')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
numeric_columns = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df = df[numeric_columns].dropna()

df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration
0,18.0,8,307.0,130.0,3504.0,12.0
1,15.0,8,350.0,165.0,3693.0,11.5
2,18.0,8,318.0,150.0,3436.0,11.0
3,16.0,8,304.0,150.0,3433.0,12.0
4,17.0,8,302.0,140.0,3449.0,10.5


## A. Hitung Korelasi

In [6]:
variabels = ['mpg','cylinders','displacement','horsepower','weight','acceleration']

korelasi_matrix = np.zeros((len(variabels), len(variabels)))

for i, v1 in enumerate(variabels):
    for j, v2 in enumerate(variabels):
        korelasi_matrix[i, j] = np.corrcoef(df[v1], df[v2])[0, 1]

print(pd.DataFrame(korelasi_matrix, index=variabels, columns=variabels))

                   mpg  cylinders  displacement  horsepower    weight  \
mpg           1.000000  -0.777618     -0.805127   -0.778427 -0.832244   
cylinders    -0.777618   1.000000      0.950823    0.842983  0.897527   
displacement -0.805127   0.950823      1.000000    0.897257  0.932994   
horsepower   -0.778427   0.842983      0.897257    1.000000  0.864538   
weight       -0.832244   0.897527      0.932994    0.864538  1.000000   
acceleration  0.423329  -0.504683     -0.543800   -0.689196 -0.416839   

              acceleration  
mpg               0.423329  
cylinders        -0.504683  
displacement     -0.543800  
horsepower       -0.689196  
weight           -0.416839  
acceleration      1.000000  


## B. Stepwise Regression (Forward Selection dengan Uji F)

In [7]:
variabel_names = ['cylinders','displacement','horsepower','weight','acceleration']
X = df[variabel_names].values
y = df['mpg'].values

remaining = list(range(len(variabel_names)))
selected = []

# Forward selection based on F-test
while remaining:
    best_f = -1
    best_idx = None
    for idx in remaining:
        X_temp = X[:, selected + [idx]]
        X_temp = np.column_stack([np.ones(len(X_temp)), X_temp])
        beta_temp = np.linalg.inv(X_temp.T @ X_temp) @ (X_temp.T @ y)
        y_pred_temp = X_temp @ beta_temp
        n = len(y)
        k = len(selected) + 1
        sse = np.sum((y - y_pred_temp)**2)
        sst = np.sum((y - np.mean(y))**2)
        ssr = sst - sse
        f = (ssr / k) / (sse / (n - k - 1))
        print(f"{variabel_names[idx]} -> F = {f:.4f}")
        if f > best_f:
            best_f = f
            best_idx = idx
    
    if best_f > 4:
        selected.append(best_idx)
        remaining.remove(best_idx)
        print(f"Menambahkan: {variabel_names[best_idx]}\n")
    else:
        break

print('Variabel terpilih:')
for i in selected:
    print('-', variabel_names[i])

X_final = X[:, selected]
X_final = np.column_stack([np.ones(len(X_final)), X_final])

beta = np.linalg.inv(X_final.T @ X_final) @ (X_final.T @ y)
ypred = X_final @ beta

cylinders -> F = 596.5650
displacement -> F = 718.6771
horsepower -> F = 599.7177
weight -> F = 878.8309
acceleration -> F = 85.1503
Menambahkan: weight

cylinders -> F = 448.3976
displacement -> F = 451.6081
horsepower -> F = 467.9102
acceleration -> F = 453.1812
Menambahkan: horsepower

cylinders -> F = 313.0616
displacement -> F = 312.0103
acceleration -> F = 311.1385
Menambahkan: cylinders

displacement -> F = 234.1913
acceleration -> F = 234.2371
Menambahkan: acceleration

displacement -> F = 186.9056
Menambahkan: displacement

Variabel terpilih:
- weight
- horsepower
- cylinders
- acceleration
- displacement


## C. Interpretasi Koefisien Regresi

In [8]:
print(f'Intercept = {beta[0]:.4f}\n')

for i,idx in enumerate(selected):
    nama = variabel_names[idx]
    print(f'{nama}: {beta[i+1]:.4f}')

Intercept = 46.2643

weight: -0.0052
horsepower: -0.0453
cylinders: -0.3979
acceleration: -0.0291
displacement: -0.0001


## D. Interpretasi R² dan Uji Statistik

In [9]:
n = len(y)
k = len(selected)

sst = np.sum((y - np.mean(y))**2)
sse = np.sum((y - ypred)**2)
ssr = sst - sse

r2 = ssr/sst
adj_r2 = 1 - ((1-r2)*(n-1)/(n-k-1))

f_stat = (ssr/k)/(sse/(n-k-1))

print(f'R² = {r2:.4f} ({r2*100:.2f}%)')
print(f'Adjusted R² = {adj_r2:.4f}')
print(f'F-statistic = {f_stat:.4f}')

R² = 0.7077 (70.77%)
Adjusted R² = 0.7039
F-statistic = 186.9056


## E. Kesimpulan dan Analisis

In [10]:
corr_target = pd.DataFrame(korelasi_matrix,index=variabels,columns=variabels)['mpg']
print('Korelasi terhadap mpg')
print(corr_target.sort_values(ascending=False))

print('\nPersamaan Regresi:')
print(f'mpg = {beta[0]:.4f}', end=' ')
for i,idx in enumerate(selected):
    print(f'+ ({beta[i+1]:.4f})*{variabel_names[idx]}', end=' ')
print()

X_std = (X - X.mean(axis=0))/X.std(axis=0)
y_std = (y - y.mean())/y.std()
Xs = np.column_stack([np.ones(len(X_std)), X_std])
beta_std = np.linalg.inv(Xs.T @ Xs) @ (Xs.T @ y_std)

print('\nKoefisien Standar:')
for i,nama in enumerate(variabel_names):
    print(nama, ':', round(beta_std[i+1],4))

Korelasi terhadap mpg
mpg             1.000000
acceleration    0.423329
cylinders      -0.777618
horsepower     -0.778427
displacement   -0.805127
weight         -0.832244
Name: mpg, dtype: float64

Persamaan Regresi:
mpg = 46.2643 + (-0.0052)*weight + (-0.0453)*horsepower + (-0.3979)*cylinders + (-0.0291)*acceleration + (-0.0001)*displacement 

Koefisien Standar:
cylinders : -0.087
displacement : -0.0011
horsepower : -0.2232
weight : -0.5645
acceleration : -0.0103
